# 🚀 Kobits Engine: Fine-Tuning Qwen 2.5 Coder on Claude Trajectories
### Distilling Claude Sonnet 4.6 Engineering Intelligence into Your Custom Model

This notebook trains **Qwen 2.5 Coder 7B (or 14B)** on the agentic trajectories harvested from Kobits.
- **Framework**: [Unsloth](https://github.com/unslothai/unsloth) (5x faster, 80% less VRAM)
- **Hardware**: Google Colab Pro (NVIDIA A100 or L4 GPU)
- **Technique**: 4-bit QLoRA with ChatML Tool-Calling Formats

## 1. Verify GPU Setup
Ensure you are running on an A100 or L4 GPU (**Runtime > Change runtime type > Hardware accelerator > GPU**).

In [ ]:
!nvidia-smi

## 2. Install Unsloth and Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

## 3. Load Base Model: Qwen 2.5 Coder 7B
We load Qwen 2.5 Coder in 4-bit precision to fit comfortably in VRAM while maintaining full precision performance.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = None # Auto detect: Float16 for T4/V100, Bfloat16 for A100/L4
load_in_4bit = True

# Choose Qwen 2.5 Coder 7B or 14B
model_name = "Qwen/Qwen2.5-Coder-14B-Instruct"  # Frontier 14B Model

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✓ Model loaded successfully!")

## 4. Attach LoRA Adapters
We add low-rank adapters to all attention and feed-forward projection layers.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✓ LoRA adapters attached!")

## 5. Load the Kobits Dataset
Upload `kobits_sft_dataset.jsonl` using the Colab file browser (left panel), then run this cell.

In [ ]:
import json
from datasets import Dataset

# Read uploaded JSONL
import os
dataset_path = "kobits_frontier_v2.jsonl" if os.path.exists("kobits_frontier_v2.jsonl") else "kobits_sft_dataset.jsonl"
print(f"Using dataset: {dataset_path}")
data_rows = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data_rows.append(json.loads(line))

# Format with tokenizer chat template
formatted_texts = []
for row in data_rows:
    messages = row.get("messages", [])
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    formatted_texts.append(formatted)

dataset = Dataset.from_dict({"text": formatted_texts})
print(f"✓ Loaded {len(dataset)} training examples!")
print("Sample:", formatted_texts[0][:250] + "...")

## 6. Train the Model with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=200,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="kobits_coder_14b_v2_checkpoints",
    ),
)

print("🚀 Starting Training on Colab Pro...")
trainer_stats = trainer.train()
print("✓ Training Complete!")

## 7. Test the Fine-Tuned Kobits Model

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "Build a modern school admission portal with online fee calculation"
messages = [
    {"role": "system", "content": "You are Kobits, an autonomous senior full-stack AI engineering agent."},
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=1024, use_cache=True, temperature=0.2)
generated_code = tokenizer.batch_decode(outputs)
print(generated_code[0])

## 8. Export Model to GGUF and Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Export 4-bit quantized GGUF for Ollama/local run
model.save_pretrained_gguf("/content/drive/MyDrive/kobits_coder_14b_v2_gguf", tokenizer, quantization_method="q4_k_m")

# Export merged 16-bit model for vLLM or Hugging Face deployment
model.save_pretrained_merged("/content/drive/MyDrive/kobits_coder_14b_v2_merged", tokenizer, save_method="merged_16bit")

print("🎉 Successfully saved your trained Kobits model to Google Drive!")